# Commodity Price Signal Backtester

This notebook builds a quantitative backtesting framework for commodity futures from scratch. We will analyze three major commodities:
- **Crude Oil (`CL=F`)**
- **Natural Gas (`NG=F`)**
- **Corn (`ZC=F`)**

We will implement two strategies: Momentum (Moving Average Crossover) and Mean-Reversion (Z-Score).

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 7)

## 1. Data Acquisition

We use `yfinance` to fetch historical daily price data for our commodity futures. The function `fetch_commodity_data` downloads the data and calculates daily returns.

In [ ]:
def fetch_commodity_data(tickers, start_date, end_date):
    """
    Fetches historical daily price data for a list of tickers.
    
    Args:
        tickers (list or str): Ticker symbol(s) to fetch.
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        
    Returns:
        pd.DataFrame: A DataFrame containing the 'Adj Close' prices for the tickers.
    """
    print(f"Fetching data for {tickers} from {start_date} to {end_date}...")
    df = yf.download(tickers, start=start_date, end=end_date)['Adj Close']
    
    # If a single ticker is passed, yfinance returns a Series instead of a DataFrame.
    # We convert it to a DataFrame to maintain consistency.
    if isinstance(df, pd.Series):
        df = df.to_frame(name=tickers if isinstance(tickers, str) else tickers[0])
        
    # Drop any rows with entirely missing data (e.g. holidays)
    df.dropna(how='all', inplace=True)
    
    # Forward fill any missing intermediate data points
    df.ffill(inplace=True)
    
    print("Data fetching complete.")
    return df